# Seq2Seq: traducción de inglés a español con un encoder-decoder

En la clase 07 el modelo recibía una secuencia y devolvía una sola etiqueta (*many-to-one*). En esta clase la salida también es una secuencia: dada una oración en inglés, el modelo genera su traducción al español palabra por palabra. Las dos oraciones tienen largos distintos y sus palabras no se corresponden una a una, así que usamos dos redes: un **encoder** que lee la oración de entrada y un **decoder** que genera la traducción.

## Introducción

### Objetivos

1. **Entender la arquitectura encoder-decoder** y qué información pasa del encoder al decoder.
2. **Preparar pares de oraciones para entrenar**: vocabularios, tokens especiales y padding por batch.
3. **Entrenar con teacher forcing** y entender por qué la pérdida de validación no es comparable con la de entrenamiento.
4. **Generar traducciones de forma autorregresiva** y medir su calidad con BLEU.

### Contenido

1. Configuración y carga del dataset.
2. División en entrenamiento, validación y test; vocabularios y tokens especiales.
3. `Dataset`, padding por batch y `DataLoader`.
4. Encoder y decoder.
5. Entrenamiento con teacher forcing.
6. Generación de traducciones y evaluación en test con BLEU.

### Sobre el conjunto de datos

Usamos pares de oraciones inglés-español de [Tatoeba](https://tatoeba.org/es/downloads), una colección de oraciones y traducciones aportadas por voluntarios, con licencia CC BY 2.0 FR. El archivo tiene unos 275,000 pares; nos quedamos con las oraciones de hasta 8 palabras en los dos idiomas.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence  # rellena secuencias de distinto largo hasta el largo de la más larga

from torchinfo import summary  # resumen de capas, formas y parámetros

import copy  # copiar los pesos de la mejor época
import random
import re  # expresiones regulares para la limpieza de texto
import time
from collections import Counter  # contar la frecuencia de cada palabra (vocabulario)

from nltk.translate.bleu_score import corpus_bleu  # métrica BLEU para traducción automática

from utils import plot_training

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
random.seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

## Carga de datos y preprocesamiento

Cada línea del archivo tiene el identificador y el texto de la oración en inglés, y el identificador y el texto de su traducción, separados por tabuladores. Pasamos el texto a minúsculas, separamos los signos de pregunta y exclamación como tokens propios, eliminamos el resto de los caracteres especiales y nos quedamos con las oraciones cortas.

In [ ]:
def clean_text(text):
    # Convertimos a minúsculas
    text = text.lower()

    # Insertamos espacios alrededor de los símbolos de puntuación que queremos conservar
    text = re.sub(r"([¿?¡!])", r" \1 ", text)

    # Eliminamos todo lo que no sea letras, números o los símbolos que queremos conservar
    # (el apóstrofo también se elimina: "i'm" queda como "i m")
    text = re.sub(r"[^a-zA-Z0-9áéíóúüñ¿?¡!]+", " ", text)

    # Eliminamos los espacios extra
    text = re.sub(r"\s+", " ", text).strip()

    return text


print(clean_text("I'm sure you're right!"))
print(clean_text("¿Cómo estás?"))

In [ ]:
DATA_PATH = "data/English-Spanish.tsv"
MAX_SENTENCE_LENGTH = 8  # máximo de palabras por oración, en los dos idiomas


def load_data(source_file, max_words):
    src_texts, trg_texts = [], []
    with open(source_file, "r", encoding="utf-8") as f:
        for line in f:
            elements = line.split("\t")
            src = clean_text(elements[1])  # inglés
            trg = clean_text(elements[3])  # español

            # Nos quedamos con los pares en los que las dos oraciones tienen hasta max_words palabras
            if len(src.split()) <= max_words and len(trg.split()) <= max_words:
                src_texts.append(src)
                trg_texts.append(trg)
    return src_texts, trg_texts


src_texts, trg_texts = load_data(DATA_PATH, MAX_SENTENCE_LENGTH)
print(f"Cantidad de pares: {len(src_texts)}")

In [ ]:
for idx in random.sample(range(len(src_texts)), 8):
    print(f"Inglés:  {src_texts[idx]}")
    print(f"Español: {trg_texts[idx]}\n")

Algunas traducciones son libres o incompletas, y muchas oraciones en inglés aparecen varias veces con traducciones distintas. Es lo esperable en un dataset aportado por voluntarios, y también refleja que una oración admite más de una traducción correcta.

### División en entrenamiento, validación y test

Como en la clase 07, separamos los conjuntos **antes** de construir los vocabularios: 80% para entrenamiento, 10% para validación y 10% para test. El test se usa una sola vez, al final.

<small>Nota: como muchas oraciones en inglés tienen varias traducciones, con una división aleatoria cerca de un cuarto de las oraciones de validación y test también aparece en entrenamiento, con otra traducción. Esto hace la evaluación algo optimista. Separar por oración en inglés (todas sus traducciones en el mismo conjunto) lo evita.</small>

In [ ]:
pairs = list(zip(src_texts, trg_texts))
train_pairs, val_pairs, test_pairs = random_split(
    pairs, [0.8, 0.1, 0.1], generator=torch.Generator().manual_seed(SEED)
)
print(f"Entrenamiento: {len(train_pairs)} | Validación: {len(val_pairs)} | Test: {len(test_pairs)}")

## Construcción de los vocabularios

Construimos un vocabulario para cada idioma, solo con el conjunto de entrenamiento. A diferencia de la clase 07, cada vocabulario mapea palabras a índices y también índices a palabras: el decoder genera índices y hay que convertirlos de nuevo en texto. Incluimos las palabras que aparecen al menos `FREQ_THRESHOLD` veces.

Además agregamos tokens especiales:

- `<PAD>`: relleno para llevar las oraciones de un batch al mismo largo (índice 0).
- `<SOS>` (*start of sentence*): primera entrada del decoder; le indica que empiece a generar.
- `<EOS>` (*end of sentence*): el decoder lo genera para indicar que la traducción terminó.
- `<UNK>` (*unknown*): palabra fuera del vocabulario. Es el `<OOV>` de la clase 07.

In [ ]:
PAD_TOKEN = "<PAD>"
SOS_TOKEN = "<SOS>"
EOS_TOKEN = "<EOS>"
UNK_TOKEN = "<UNK>"
FREQ_THRESHOLD = 3  # frecuencia mínima para incluir una palabra en el vocabulario


class Vocab:
    def __init__(self):
        # mapea palabras a índices
        self.word2index = {}
        # mapea índices a palabras
        self.index2word = {}
        # autonumeración de índices
        self.index = 0

        # Tokens especiales
        self.add_special_tokens()

    def add_special_tokens(self):
        self.add_word(PAD_TOKEN)
        self.add_word(SOS_TOKEN)
        self.add_word(EOS_TOKEN)
        self.add_word(UNK_TOKEN)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.index
            self.index2word[self.index] = word
            self.index += 1

    def build_vocab(self, sentences, min_freq=1):
        word_counter = Counter()
        for sentence in sentences:
            for word in sentence.split():
                word_counter[word] += 1

        # Filtrar palabras que no alcanzan la frecuencia mínima
        words = [word for word, count in word_counter.items() if count >= min_freq]

        # Agregar palabras filtradas al vocabulario
        for word in words:
            self.add_word(word)

    def __len__(self):
        return len(self.word2index)

    def __getitem__(self, key):
        if isinstance(key, int):
            return self.index2word.get(key, UNK_TOKEN)
        if isinstance(key, str):
            return self.word2index.get(key, self.word2index[UNK_TOKEN])


# Construimos los vocabularios solo con el conjunto de entrenamiento
SRC_VOCAB = Vocab()
TRG_VOCAB = Vocab()

SRC_VOCAB.build_vocab([src for src, _ in train_pairs], min_freq=FREQ_THRESHOLD)
TRG_VOCAB.build_vocab([trg for _, trg in train_pairs], min_freq=FREQ_THRESHOLD)

SRC_VOCAB_SIZE = len(SRC_VOCAB)
TRG_VOCAB_SIZE = len(TRG_VOCAB)

print(f"Tamaño del vocabulario en inglés: {SRC_VOCAB_SIZE}")
print(f"Tamaño del vocabulario en español: {TRG_VOCAB_SIZE}")

Algunos ejemplos de uso de los vocabularios:

- `SRC_VOCAB["hello"]`: índice de la palabra "hello" en el vocabulario de origen.
- `TRG_VOCAB["hola"]`: índice de la palabra "hola" en el vocabulario de destino.
- `TRG_VOCAB["palabra_no_existente"]`: índice de `<UNK>`, porque la palabra no está en el vocabulario.
- `TRG_VOCAB[10]`: palabra con índice 10 en el vocabulario de destino.
- `TRG_VOCAB[3]`: palabra con índice 3 (es `<UNK>`).
- `TRG_VOCAB[PAD_TOKEN]`: índice del token de padding.

In [ ]:
print(SRC_VOCAB["hello"])
print(TRG_VOCAB["hola"])
print(TRG_VOCAB["palabra_no_existente"])
print(TRG_VOCAB[10])
print(TRG_VOCAB[3])
print(TRG_VOCAB[PAD_TOKEN])

In [ ]:
# Función para codificar una frase (strings -> índices)
def encode_sentence(sentence, vocab):
    return [vocab[word] for word in sentence.split()]


print(encode_sentence("hello world", SRC_VOCAB))
print(encode_sentence("hola mundo extraterrestre", TRG_VOCAB))

In [ ]:
# Función para decodificar una secuencia de índices (índices -> strings)
def decode_sentence(indices, vocab):
    if isinstance(indices, torch.Tensor):  # en caso que nos pasen un tensor
        indices = indices.tolist()
    return " ".join(
        [
            vocab[idx]
            for idx in indices
            if idx != vocab[PAD_TOKEN]
            and idx != vocab[EOS_TOKEN]
            and idx != vocab[SOS_TOKEN]
        ]
    )


print(
    decode_sentence(
        [
            TRG_VOCAB[SOS_TOKEN],
            10,
            11,
            12,
            TRG_VOCAB[EOS_TOKEN],
            TRG_VOCAB[PAD_TOKEN],
            TRG_VOCAB[PAD_TOKEN],
        ],
        TRG_VOCAB,
    )
)

## Dataset de traducción

Cada oración se convierte en índices de su vocabulario. Al target le agregamos `<SOS>` al principio y `<EOS>` al final.

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs):
        super(TranslationDataset, self).__init__()
        self.pairs = pairs  # lista de pares (oración en inglés, oración en español)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # TODO:
        #   1. obtener el par (source_sentence, target_sentence) en la posición idx
        #   2. codificar cada oración con encode_sentence y su vocabulario (SRC_VOCAB, TRG_VOCAB)
        #   3. agregar al target <SOS> al principio y <EOS> al final
        #   4. devolver dos tensores de tipo torch.long: (source, target)
        pass


train_dataset = TranslationDataset(train_pairs)
val_dataset = TranslationDataset(val_pairs)
test_dataset = TranslationDataset(test_pairs)

In [ ]:
# Probamos algunos ejemplos de entrenamiento
for idx in range(3):
    x, y = train_dataset[idx]
    print(f"SRC:           {decode_sentence(x, SRC_VOCAB)}")
    print(f"SRC (índices): {x}")
    print(f"TRG:           {decode_sentence(y, TRG_VOCAB)}")
    print(f"TRG (índices): {y}\n")

### Padding por batch

Para apilar las oraciones de un batch en un tensor, todas tienen que tener el mismo largo. A diferencia de la clase 07, no fijamos un largo único para todo el dataset: cada batch se rellena hasta su oración más larga con [`pad_sequence`](https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pad_sequence.html). Para eso usamos el argumento `collate_fn` del `DataLoader`, una función que recibe la lista de ejemplos del batch y arma los tensores.

<img src="https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/seq2seq_padding.png" width="900" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

- Source con padding a la izquierda: como en las clases 06 y 07, las palabras reales quedan al final y el estado final del encoder se calcula justo después de la última palabra. Ese estado es lo único que recibe el decoder.
- Target con padding a la derecha: `<SOS>` queda en la posición 0 de todas las oraciones, que es donde el decoder empieza a generar. Los `<PAD>` quedan después de `<EOS>` y la pérdida los ignora (`ignore_index`).

La entrada y el objetivo del decoder son el mismo target desplazado una posición: en cada paso el decoder recibe un token y tiene que predecir el siguiente.

In [ ]:
seq_1 = torch.tensor([1, 2, 3, 4])
seq_2 = torch.tensor([1, 2, 3])
seq_3 = torch.tensor([1, 2, 3, 4, 5, 6])

print(pad_sequence([seq_1, seq_2, seq_3], batch_first=True, padding_value=0, padding_side="left"))
print(pad_sequence([seq_1, seq_2, seq_3], batch_first=True, padding_value=0, padding_side="right"))

In [ ]:
def collate_fn(batch):
    # batch: lista de pares (source, target) devueltos por TranslationDataset
    sources, targets = zip(*batch)

    # TODO:
    #   1. rellenar sources con pad_sequence: batch_first=True, padding_value=SRC_VOCAB[PAD_TOKEN], padding_side="left"
    #   2. rellenar targets con pad_sequence: batch_first=True, padding_value=TRG_VOCAB[PAD_TOKEN], padding_side="right"
    #   3. devolver (sources_padded, targets_padded)
    pass

In [ ]:
BATCH_SIZE = 512

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=NUM_WORKERS
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=NUM_WORKERS
)

In [ ]:
x, y = next(iter(train_loader))
print(f"Source batch shape: {x.shape}")  # (batch, largo de la oración más larga del batch)
print(f"Target batch shape: {y.shape}")  # hasta MAX_SENTENCE_LENGTH + 2, por <SOS> y <EOS>

## Modelo Seq2Seq

El encoder y el decoder son LSTM de dos capas (clase 07). El estado final del encoder, $(h, c)$ de cada capa, es el estado inicial del decoder. Por eso las dos LSTM tienen que tener el mismo `HIDDEN_DIM` y la misma cantidad de capas.

<img src="https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/seq2seq_train_vs_inference.png" width="900" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

Toda la oración de entrada tiene que caber en ese estado de tamaño fijo: es un **cuello de botella**. En oraciones largas el decoder pierde información, y ese es el problema que resuelve el mecanismo de atención de la clase 09.

In [ ]:
# Hiperparámetros del modelo y del entrenamiento
EMBEDDING_DIM = 128
HIDDEN_DIM = 512
N_LAYERS = 2
DROPOUT = 0.5

LR = 1e-3
EPOCHS = 30

INITIAL_TEACHER_FORCING_RATIO = 1.0
TEACHER_FORCING_DECAY = 0.95
MIN_TEACHER_FORCING_RATIO = 0.3

### Encoder

- La oración de entrada pasa por una capa de embedding.
- Una LSTM de `N_LAYERS` capas recorre la secuencia de embeddings.
- El encoder devuelve el último estado de cada capa, `hidden` y `cell`.

Descartamos `outputs`, los estados de la última capa en todos los pasos. Con el mecanismo de atención (clase 09), el decoder también los usa.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# Encoder                                  [2, 512, 512]             --
# ├─Embedding: 1-1                         [512, 8, 128]             1,130,624
# ├─Dropout: 1-2                           [512, 8, 128]             --
# ├─LSTM: 1-3                              [512, 8, 512]             3,416,064
# Total params: 4,546,688


class Encoder(nn.Module):
    def __init__(self, emb_dim, hidden_dim, n_layers, dropout):
        super(Encoder, self).__init__()
        # TODO: definir las capas como atributos:
        #   1. self.embedding: nn.Embedding de SRC_VOCAB_SIZE a emb_dim, con padding_idx=SRC_VOCAB[PAD_TOKEN]
        #   2. self.dropout: nn.Dropout(dropout)
        #   3. self.lstm: nn.LSTM de emb_dim a hidden_dim, con n_layers capas, batch_first=True y dropout=dropout
        pass

    def forward(self, src):
        # src: (batch, src_len) con índices enteros
        # TODO:
        #   1. embedding + dropout -> (batch, src_len, emb_dim)
        #   2. lstm -> outputs, (hidden, cell); hidden y cell: (n_layers, batch, hidden_dim)
        #   3. devolver hidden, cell (outputs no se usa)
        pass


summary(
    Encoder(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT),
    input_size=(BATCH_SIZE, MAX_SENTENCE_LENGTH),
    dtypes=[torch.long],
)

### Decoder

El decoder genera la traducción un token por vez. En cada paso:

- Recibe un token (en el paso 0, `<SOS>`) y el estado anterior `(hidden, cell)`. En el paso 0 ese estado es el del encoder.
- Pasa el token por su embedding y por la LSTM, que actualiza el estado.
- Una capa lineal convierte la salida de la LSTM en logits sobre todo el vocabulario de destino: en cada paso el decoder resuelve una clasificación entre `TRG_VOCAB_SIZE` clases.

A diferencia de la clase 07, la LSTM no recibe la secuencia completa de una vez, sino un token por vez dentro de un bucle. En inferencia es necesario: la entrada de cada paso es la predicción del paso anterior, que todavía no existe cuando empieza el bucle.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# Decoder                                  [512, 13402]              --
# ├─Embedding: 1-1                         [512, 1, 128]             1,715,456
# ├─Dropout: 1-2                           [512, 1, 128]             --
# ├─LSTM: 1-3                              [512, 1, 512]             3,416,064
# ├─Linear: 1-4                            [512, 13402]              6,875,226
# Total params: 12,006,746


class Decoder(nn.Module):
    def __init__(self, emb_dim, hidden_dim, n_layers, dropout):
        super(Decoder, self).__init__()
        # TODO: definir las capas como atributos:
        #   1. self.embedding: nn.Embedding de TRG_VOCAB_SIZE a emb_dim, con padding_idx=TRG_VOCAB[PAD_TOKEN]
        #   2. self.dropout: nn.Dropout(dropout)
        #   3. self.lstm: nn.LSTM de emb_dim a hidden_dim, con n_layers capas, batch_first=True y dropout=dropout
        #   4. self.fc_out: nn.Linear de hidden_dim a TRG_VOCAB_SIZE
        pass

    def forward(self, input_token, hidden, cell):
        # input_token: (batch,) -> un solo token por cada oración del batch
        # hidden y cell: (n_layers, batch, hidden_dim)
        # TODO:
        #   1. input_token.unsqueeze(1) -> (batch, 1): la LSTM espera una secuencia, de largo 1
        #   2. embedding + dropout -> (batch, 1, emb_dim)
        #   3. lstm con el estado (hidden, cell) -> output, (hidden, cell); output: (batch, 1, hidden_dim)
        #   4. fc_out sobre output.squeeze(1) -> logits (batch, TRG_VOCAB_SIZE)
        #   5. devolver prediction, hidden, cell
        pass


summary(
    Decoder(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT),
    input_size=[
        (BATCH_SIZE,),
        (N_LAYERS, BATCH_SIZE, HIDDEN_DIM),
        (N_LAYERS, BATCH_SIZE, HIDDEN_DIM),
    ],
    dtypes=[torch.long, torch.float, torch.float],
)

## Entrenamiento

En cada paso, el decoder resuelve una clasificación entre todas las palabras del vocabulario de destino, así que usamos `nn.CrossEntropyLoss`. Con `ignore_index` la pérdida no cuenta las posiciones de `<PAD>`. Con `label_smoothing=0.05` el objetivo asigna 0.95 a la palabra correcta y reparte el resto entre las demás: una oración admite varias traducciones, así que conviene que el modelo no esté seguro por completo de una sola palabra.

El optimizador recibe los parámetros del encoder y del decoder, que se entrenan juntos.

In [ ]:
torch.manual_seed(SEED)
encoder = Encoder(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).to(DEVICE)
decoder = Decoder(EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).to(DEVICE)

criterion = nn.CrossEntropyLoss(ignore_index=TRG_VOCAB[PAD_TOKEN], label_smoothing=0.05)
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=LR)

### Teacher forcing

Durante el entrenamiento, el decoder podría recibir en cada paso su propia predicción anterior, como en inferencia. Al principio esas predicciones son casi aleatorias: un error temprano arrastra al resto de la oración y el aprendizaje es lento. Con **teacher forcing**, en cada paso el decoder recibe el token correcto del target, aunque haya predicho otro (fila de arriba del diagrama).

Entrenar siempre así genera un desajuste: en inferencia el modelo solo ve sus propias predicciones, y nunca aprendió a seguir después de un error propio (*exposure bias*). Por eso bajamos la probabilidad de teacher forcing época a época, una idea conocida como *scheduled sampling* (Bengio et al., 2015): empieza en `INITIAL_TEACHER_FORCING_RATIO`, se multiplica por `TEACHER_FORCING_DECAY` en cada época y no baja de `MIN_TEACHER_FORCING_RATIO`.

En `process_batch`, la pérdida se calcula en cada paso del decoder y se promedia sobre los pasos. El sorteo de teacher forcing se hace en cada paso, para todo el batch a la vez. Después de recorrer todos los pasos se hace un único `backward()`.

In [ ]:
def process_batch(src, trg, encoder, decoder, criterion, teacher_forcing_ratio):
    trg_input = trg[:, :-1]  # entrada del decoder: sin el último token
    trg_target = trg[:, 1:]  # objetivo: sin <SOS>
    trg_seq_len = trg_input.size(1)

    # TODO:
    #   1. pasar src por el encoder -> hidden, cell (estado inicial del decoder)
    #   2. input_token = trg_input[:, 0] (paso 0: <SOS>)
    #   3. para cada paso t en range(trg_seq_len):
    #      a. decoder(input_token, hidden, cell) -> output, hidden, cell
    #      b. sumar criterion(output, trg_target[:, t]) a la pérdida del batch
    #      c. con probabilidad teacher_forcing_ratio (random.random() < teacher_forcing_ratio),
    #         input_token = trg_target[:, t]; si no, input_token = output.argmax(1).detach()
    #   4. devolver la pérdida promediada sobre los pasos (dividir por trg_seq_len)
    pass

In [ ]:
def train_epoch(encoder, decoder, dataloader, optimizer, criterion, teacher_forcing_ratio):
    encoder.train()
    decoder.train()
    epoch_loss = 0

    # TODO: para cada batch (src, trg) del dataloader:
    #   1. mover src y trg a DEVICE
    #   2. optimizer.zero_grad()
    #   3. batch_loss = process_batch(...) con el teacher_forcing_ratio recibido
    #   4. backward y optimizer.step()
    #   5. acumular batch_loss.item() en epoch_loss

    return epoch_loss / len(dataloader)

In [ ]:
def evaluate(encoder, decoder, dataloader, criterion):
    encoder.eval()
    decoder.eval()
    epoch_loss = 0

    # TODO: igual que train_epoch, pero:
    #   1. dentro de torch.no_grad(), sin zero_grad, backward ni step
    #   2. con teacher_forcing_ratio=0.0: el decoder recibe sus propias predicciones, como en inferencia

    return epoch_loss / len(dataloader)

In [ ]:
def train(
    encoder,
    decoder,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    n_epochs,
    initial_teacher_forcing_ratio=INITIAL_TEACHER_FORCING_RATIO,
    teacher_forcing_decay=TEACHER_FORCING_DECAY,
    min_teacher_forcing_ratio=MIN_TEACHER_FORCING_RATIO,
):
    train_losses, val_losses = [], []
    best_val_loss, best_weights = float("inf"), None
    teacher_forcing_ratio = initial_teacher_forcing_ratio

    for epoch in range(n_epochs):
        start = time.time()
        train_loss = train_epoch(encoder, decoder, train_loader, optimizer, criterion, teacher_forcing_ratio)
        val_loss = evaluate(encoder, decoder, val_loader, criterion)
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # Guardamos los pesos de la época con menor pérdida de validación
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = (copy.deepcopy(encoder.state_dict()), copy.deepcopy(decoder.state_dict()))

        print(
            f"Época {epoch + 1:02d} | TF: {teacher_forcing_ratio:.2f} | Train Loss: {train_loss:.4f} "
            f"| Val Loss: {val_loss:.4f} | {time.time() - start:.0f} s"
        )
        teacher_forcing_ratio = max(min_teacher_forcing_ratio, teacher_forcing_ratio * teacher_forcing_decay)

    # El modelo queda con los pesos de la mejor época, no de la última
    encoder.load_state_dict(best_weights[0])
    decoder.load_state_dict(best_weights[1])
    return train_losses, val_losses

In [ ]:
train_losses, val_losses = train(
    encoder, decoder, train_loader, val_loader, optimizer, criterion, n_epochs=EPOCHS
)

In [ ]:
plot_training(train_losses, val_losses, mark_min=True)

La pérdida de validación queda siempre por encima de la de entrenamiento (en nuestra ejecución, 2.51 en la mejor época contra 1.82 de entrenamiento al final). No se trata de sobreajuste, porque las dos curvas siguen bajando. Las dos pérdidas se miden en condiciones distintas: en entrenamiento, el decoder recibe el token correcto con probabilidad `TF` y un error no se propaga a los pasos siguientes; en validación (`teacher_forcing_ratio=0.0`) recibe sus propias predicciones, como en inferencia, y un error temprano afecta al resto de la oración.

Además, muchas oraciones admiten varias traducciones válidas y la pérdida compara con una sola referencia, por eso ninguna de las dos curvas se acerca a cero. En la época 30 la pérdida de validación todavía baja: con más épocas el modelo probablemente mejore un poco.

## Generación de traducciones

Para traducir no hay target: el decoder empieza con `<SOS>` y en cada paso recibe su propia predicción anterior (fila de abajo del diagrama). Esta forma de generar, un token por vez a partir de los anteriores, se llama **autorregresiva**. En cada paso elegimos la palabra más probable (`argmax`): es la decodificación **greedy**. La generación termina cuando el modelo produce `<EOS>` o al llegar a `max_len` tokens.

<small>Elegir en cada paso la palabra más probable no garantiza la oración más probable. *Beam search* mantiene en cada paso las k oraciones parciales más probables y suele dar mejores traducciones.</small>

In [ ]:
def generate_sequence(src, encoder, decoder, max_len):
    # src: (1, src_len), una sola oración
    encoder.eval()
    decoder.eval()
    generated_tokens = []
    with torch.no_grad():
        # TODO:
        #   1. pasar src por el encoder -> hidden, cell
        #   2. input_token = tensor con el índice de <SOS>, de forma (1,), en DEVICE
        #   3. repetir hasta max_len veces:
        #      a. decoder(input_token, hidden, cell) -> output, hidden, cell
        #      b. top1 = output.argmax(1).item() (greedy)
        #      c. si top1 es <EOS>, terminar
        #      d. agregar top1 a generated_tokens y usarlo como próxima entrada
        pass

    return generated_tokens

In [ ]:
def translate_sentence(encoder, decoder, sentence, max_len):
    sentence = clean_text(sentence)
    sentence_indexes = encode_sentence(sentence, SRC_VOCAB)
    sentence_tensor = torch.tensor(sentence_indexes).unsqueeze(0).to(DEVICE)  # (1, src_len)
    generated_tokens = generate_sequence(sentence_tensor, encoder, decoder, max_len)
    return decode_sentence(generated_tokens, TRG_VOCAB)

In [ ]:
sentences = [
    "I am hungry",
    "I am tired",
    "I'm sad",
    "every time I study, I get sleepy",
    "I am going to the beach",
    "I'm going to the movies",
    "I don't know what to do",
    "I love deep learning",
    "I can't open the door",
    "you can go if you want to",
    "where does all this come from?",
    "I can't believe it",
    "Do you speak Italian?",
    "Do you want to go to the movies?",
    "this is my favorite song",
    "see you later",
    "have a nice day",
    "let's grab a coffee sometime",
    "she's my best friend",
    "My class is in 30 minutes.",
    "This is a very long sentence, let's see how the model handles it.",
]

for sentence in sentences:
    print(f"Inglés:     {sentence}")
    print(f"Traducción: {translate_sentence(encoder, decoder, sentence, max_len=15)}\n")

Las oraciones cortas y frecuentes se traducen bien. Los errores muestran las limitaciones del modelo:

- Repite palabras ("amiga amiga", "café café café"), un problema conocido de los modelos seq2seq con decodificación greedy.
- Las palabras poco frecuentes o fuera del vocabulario se traducen mal ("deep learning" → "aprender vacaciones").
- La oración larga tiene más de `MAX_SENTENCE_LENGTH` palabras, un largo que el modelo no vio en el entrenamiento, y toda tiene que pasar por el estado $(h, c)$. La traducción se degrada por completo: es el cuello de botella que motiva la atención de la clase 09.

In [ ]:
translate_sentence(encoder, decoder, "we need attention!", max_len=15)

## Evaluación en test

Para medir la calidad de las traducciones usamos **BLEU** ([Papineni et al., 2002](https://aclanthology.org/P02-1040/)), la métrica clásica de traducción automática. BLEU compara cada traducción generada con la traducción de referencia: mide qué proporción de sus n-gramas (secuencias de 1 a 4 palabras) aparece en la referencia, y penaliza las traducciones más cortas que la referencia. Va de 0 a 100.

BLEU tiene limitaciones: una traducción correcta con otras palabras obtiene un puntaje bajo, y en este dataset cada oración tiene una sola referencia. Por eso conviene mirarlo junto con algunos ejemplos.

Traducimos una muestra de `N_BLEU` oraciones del conjunto de test, una sola vez, con el modelo final. La generación es de a una oración, así que no usamos el test completo para que la celda tarde poco.

In [ ]:
N_BLEU = 2000

references, hypotheses = [], []
for src, trg in list(test_pairs)[:N_BLEU]:
    prediction = translate_sentence(encoder, decoder, src, max_len=MAX_SENTENCE_LENGTH + 2)
    references.append([trg.split()])  # lista de referencias por oración (acá, una sola)
    hypotheses.append(prediction.split())

print(f"BLEU en test ({len(hypotheses)} oraciones): {100 * corpus_bleu(references, hypotheses):.1f}\n")

for (src, trg), hyp in list(zip(test_pairs, hypotheses))[:8]:
    print(f"Inglés:     {src}")
    print(f"Referencia: {trg}")
    print(f"Modelo:     {' '.join(hyp)}\n")

En nuestra ejecución el BLEU en test es 28.2. Los ejemplos muestran las dos caras de la métrica: "está enamorado de ella" es una traducción correcta (en español el sujeto puede omitirse), pero BLEU la penaliza por no incluir "él"; en cambio, "estuve aquí aquí" suma puntos por las palabras que comparte con la referencia, aunque repite una.

## Conclusiones

- Un modelo seq2seq usa dos redes: el encoder resume la oración de entrada en su estado final $(h, c)$, y el decoder genera la traducción a partir de ese estado, un token por vez.
- El source se rellena a la izquierda, como en las clases 06 y 07. El target lleva `<SOS>` y `<EOS>` y se rellena a la derecha, y la pérdida ignora los `<PAD>`.
- Teacher forcing acelera el entrenamiento, pero el modelo no ve sus propios errores. Bajarlo época a época reduce la diferencia con la inferencia.
- La pérdida de validación se mide sin teacher forcing, así que no es comparable con la de entrenamiento.
- La generación es autorregresiva y greedy. BLEU mide la coincidencia de n-gramas con una referencia y conviene mirarlo junto con ejemplos.
- Toda la oración pasa por un estado de tamaño fijo, y en oraciones largas la traducción se degrada. La atención (clase 09) permite que el decoder consulte todos los estados del encoder en cada paso.

## Ejercicios

1. **Sin decaimiento de teacher forcing**: entrenar con `TEACHER_FORCING_DECAY = 1.0` (teacher forcing siempre) y con `INITIAL_TEACHER_FORCING_RATIO = 0.0` (nunca). Comparar las curvas de pérdida y las traducciones de ejemplo.
2. **Oraciones más largas**: subir `MAX_SENTENCE_LENGTH` a 12 o 15. ¿Cómo cambian el BLEU y las traducciones de oraciones largas? Relacionarlo con el cuello de botella.
3. **Beam search**: implementar una búsqueda en haz con k = 3 en `generate_sequence` y comparar con la decodificación greedy.
4. **Separar por oración en inglés**: hacer la división de modo que todas las traducciones de una oración queden en el mismo conjunto. ¿Cambia el BLEU?
5. **Adelanto de la atención**: en el encoder, devolver también `outputs`. ¿Qué forma tiene? ¿Cómo podría usarlo el decoder en cada paso?

#### Lectura adicional

- Sutskever et al. (2014), [*Sequence to Sequence Learning with Neural Networks*](https://arxiv.org/abs/1409.3215): el modelo encoder-decoder con LSTM para traducción.
- Cho et al. (2014), [*Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*](https://arxiv.org/abs/1406.1078): la arquitectura encoder-decoder y la GRU.
- Bengio et al. (2015), [*Scheduled Sampling for Sequence Prediction with Recurrent Neural Networks*](https://arxiv.org/abs/1506.03099): el decaimiento de teacher forcing.
- Papineni et al. (2002), [*BLEU: a Method for Automatic Evaluation of Machine Translation*](https://aclanthology.org/P02-1040/): la métrica BLEU.
- Bahdanau et al. (2015), [*Neural Machine Translation by Jointly Learning to Align and Translate*](https://arxiv.org/abs/1409.0473): el mecanismo de atención, que resuelve el cuello de botella (clase 09).